In [21]:
# adata=sc.read("out/10x_sect1_stf_results_tfa_only.h5ad")
# adata_raw=sc.read("data/10xGenomics/sect1.h5ad")
# adata.obs[['subtype', 'pathology', 'sample', 'replicate', 'source', 'ER', 'HER2', 'PR']]=adata_raw.obs[['subtype', 'pathology', 'sample', 'replicate', 'source', 'ER', 'HER2', 'PR']].loc[adata.obs_names]
# adata.obsm['celltype_minor']=adata_raw.obsm['celltype_minor'].loc[adata.obs_names]
# adata.obsm['celltype_major']=adata_raw.obsm['celltype_major'].loc[adata.obs_names]
# adata.write("out/10x_sect1_stf_results_tfa_only.h5ad")

In [2]:
import scanpy as sc
from glob import glob


In [27]:
adatas={x: sc.read_h5ad(x) for x in glob("out/*stf_results_tfa_only.h5ad")}
for sample, adata in adatas.items():
    if "Cancer Epithelial" in adata.obsm["celltype_major"]:
        adata.obsm["celltype_major"]["Epithelial"]=adata.obsm["celltype_major"]["Normal Epithelial"]+adata.obsm["celltype_major"]["Cancer Epithelial"]
        adata.obsm["celltype_major"].drop(columns=["Normal Epithelial", "Cancer Epithelial"], inplace=True)
    adata.obsm["celltype_major"]["Lymphocytes"]=adata.obsm["celltype_major"]["T-cells"]+adata.obsm["celltype_major"]["B-cells"]
    adata.obsm["celltype_major"].drop(columns=["T-cells", "B-cells"], inplace=True)

In [32]:
import anndata as ad
adata=ad.concat(adatas)


In [34]:
subtype_data = adata.obs.groupby("sample").first()['subtype']
for subtype in subtype_data.unique():
    j = 0
    for i, x in enumerate(subtype_data):
        if x == subtype:
            subtype_data.iloc[i] = x + "_" + str(j)
            j = j + 1
adata.obs['sample_id'] = adata.obs['sample'].map(subtype_data)

In [35]:
adata.obs['sample_id']

B1_10x13    HER2_1
B1_10x14    HER2_1
B1_10x15    HER2_1
B1_10x16    HER2_1
B1_10x17    HER2_1
             ...  
H2_9x30     HER2_7
H2_9x31     HER2_7
H2_9x32     HER2_7
H2_9x33     HER2_7
H2_9x34     HER2_7
Name: sample_id, Length: 32781, dtype: object

In [33]:
adata.obs

import statsmodels.api as sm
ct_tf_df=[]

for sample in adatas_all.obs['sample'].unique():
    print(sample)
    adata=adatas_all[adatas_all.obs['sample']==sample]
    source=adata.obs["source"].iloc[0]
    subtype=adata.obs['subtype'].iloc[0]

    Y=adata.to_df()
    Y=Y-Y.mean()
    Y=Y/Y.std()

    for tf in adata.obsm['tfa'].columns:
            results = sm.OLS(adata.obsm['tfa'][tf],adata.obsm['celltype_major']).fit()
            for ct in adata.obsm['celltype_major'].columns:
                p=results.pvalues[ct]
                coef=results.params[ct]

                ct_tf_df.append([source, sample, subtype, tf, ct, coef, p])

pd.DataFrame(ct_tf_df, columns=["source", "sample", "subtype", "tf", "ct", "coef", "p"]).to_csv("ct_df.csv")
ct_tf_df=pd.DataFrame(ct_tf_df, columns=["source", "sample", "subtype", "tf", "ct", "coef", "p"])
ct_tf_df["negative_log_p"]=-np.log10(ct_tf_df["p"]+1e-10)
ct_tf_df=ct_tf_df.sort_values(by=['source', 'sample', 'tf', 'ct'])

,pathology,sample,replicate,source,ER,HER2,PR,subtype,n_counts,pixel_intensity,pred_cor_stl,pred_cor_stan
B1_10x13,breast glands,B,B1,Andersson2021,False,True,True,HER2,399.0,111.064561,0.049879,0.037218
B1_10x14,connective tissue,B,B1,Andersson2021,False,True,True,HER2,229.0,108.254241,0.019114,0.018338
B1_10x15,connective tissue,B,B1,Andersson2021,False,True,True,HER2,237.0,86.301124,0.020301,0.033159
B1_10x16,connective tissue,B,B1,Andersson2021,False,True,True,HER2,365.0,74.581538,0.036007,0.036941
B1_10x17,connective tissue,B,B1,Andersson2021,False,True,True,HER2,238.0,86.933718,0.046833,0.048548
...,...,...,...,...,...,...,...,...,...,...,...,...
H2_9x30,,H,H2,Andersson2021,False,True,False,HER2,2736.0,124.939655,0.045534,0.044242
H2_9x31,,H,H2,Andersson2021,False,True,False,HER2,7243.0,126.663674,0.049983,0.039465
H2_9x32,,H,H2,Andersson2021,False,True,False,HER2,1742.0,138.471336,0.038915,-0.000136
H2_9x33,,H,H2,Andersson2021,False,True,False,HER2,2743.0,123.722919,0.045297,0.032054


In [26]:
adata.uns['spatial']

{'B1': {'images': {'hires': array([[[215, 206, 223],
           [215, 206, 223],
           [215, 206, 223],
           ...,
           [216, 208, 221],
           [216, 208, 221],
           [216, 208, 221]],
   
          [[215, 206, 223],
           [215, 206, 223],
           [215, 206, 223],
           ...,
           [216, 208, 221],
           [216, 208, 221],
           [216, 208, 221]],
   
          [[215, 206, 223],
           [215, 206, 223],
           [215, 206, 223],
           ...,
           [216, 208, 221],
           [216, 208, 221],
           [216, 208, 221]],
   
          ...,
   
          [[216, 207, 224],
           [216, 207, 224],
           [216, 207, 224],
           ...,
           [218, 210, 223],
           [218, 210, 223],
           [218, 210, 223]],
   
          [[216, 207, 224],
           [216, 207, 224],
           [216, 207, 224],
           ...,
           [218, 210, 223],
           [218, 210, 223],
           [218, 210, 223]],
   
          [

In [20]:
adata_raw.obs

,in_tissue,array_row,array_col,subtype,pathology,sample,replicate,source,ER,HER2,PR
AAACAAGTATCTCCCA-1,1,50,102,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
AAACACCAATAACTGC-1,1,59,19,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
AAACAGAGCGACTCCT-1,1,14,94,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
AAACAGGGTCTATATT-1,1,47,13,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
AAACAGTGTTCCTGGG-1,1,73,43,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
...,...,...,...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
TTGTTTCACATCCAGG-1,1,58,42,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
TTGTTTCATTAGTCTA-1,1,60,30,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
TTGTTTCCATACAACT-1,1,45,27,HER2,NA,V1_Breast_Cancer_Block_A,Section_1,10x_Genomics,True,True,False
